In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds

months = np.arange(1, 13)
variable_cost = np.array([250, 300, 700, 500, 450, 90, 200, 400, 200, 250, 350, 150], dtype=float)
demand = np.array([1000, 2000, 5000, 4000, 3000, 500, 1000, 8000, 4000, 2000, 6000, 3000], dtype=float)

CAPACITY = 10_000
HOLDING_COST = 30

data = pd.DataFrame({
    "Month": months,
    "Variable cost (₹/unit)": variable_cost.astype(int),
    "Demand (units)": demand.astype(int)
})
display(data)


,Month,Variable cost (₹/unit),Demand (units)
0,1,250,1000
1,2,300,2000
2,3,700,5000
3,4,500,4000
4,5,450,3000
5,6,90,500
6,7,200,1000
7,8,400,8000
8,9,200,4000
9,10,250,2000


In [2]:
def solve_production_plan(setup_cost=None):
    """Solve the 12-month production planning model.

    setup_cost=None: ignore fixed costs (LP).
    setup_cost=number: include binary monthly setup decisions (MILP).
    """
    use_binary = setup_cost is not None
    n_months = 12
    n_vars = 36 if use_binary else 24  # x[0:12], I[12:24], optional y[24:36]

    # Objective: variable production cost + holding cost + optional setup cost.
    objective = np.zeros(n_vars)
    objective[:12] = variable_cost
    objective[12:24] = HOLDING_COST
    objective[23] = 0  # I_12 is zero; no ending inventory/holding cost
    if use_binary:
        objective[24:36] = setup_cost

    rows, lower_constraints, upper_constraints = [], [], []

    # Inventory balance: x_t + I_(t-1) - I_t = demand_t.
    # For t=0, initial inventory I_0=0.
    for t in range(n_months):
        row = np.zeros(n_vars)
        row[t] = 1
        row[12 + t] = -1
        if t > 0:
            row[12 + t - 1] = 1
        rows.append(row)
        lower_constraints.append(demand[t])
        upper_constraints.append(demand[t])

    # Linking constraint: x_t <= capacity * y_t.
    if use_binary:
        for t in range(n_months):
            row = np.zeros(n_vars)
            row[t] = 1
            row[24 + t] = -CAPACITY
            rows.append(row)
            lower_constraints.append(-np.inf)
            upper_constraints.append(0)

    lower_bounds = np.zeros(n_vars)
    upper_bounds = np.full(n_vars, np.inf)
    upper_bounds[:12] = CAPACITY
    upper_bounds[23] = 0  # ending inventory I_12 = 0
    if use_binary:
        upper_bounds[24:36] = 1

    integrality = np.zeros(n_vars, dtype=int)
    if use_binary:
        integrality[24:36] = 1  # binary integer variables with bounds [0, 1]

    result = milp(
        c=objective,
        integrality=integrality,
        bounds=Bounds(lower_bounds, upper_bounds),
        constraints=LinearConstraint(
            np.asarray(rows),
            np.asarray(lower_constraints),
            np.asarray(upper_constraints)
        ),
        options={"time_limit": 60}
    )

    if not result.success:
        raise RuntimeError(f"Optimization failed: {result.message}")

    production = np.rint(result.x[:12]).astype(int)
    inventory = np.rint(result.x[12:24]).astype(int)
    setup = np.rint(result.x[24:36]).astype(int) if use_binary else (production > 0).astype(int)

    # Recalculate cost components from the returned solution.
    production_cost = float(np.dot(variable_cost, production))
    holding_cost = float(HOLDING_COST * inventory[:11].sum())
    fixed_cost_total = float(setup_cost * setup.sum()) if use_binary else 0.0

    solution = pd.DataFrame({
        "Month": months,
        "Demand (units)": demand.astype(int),
        "Production (units)": production,
        "Ending inventory (units)": inventory,
        "Variable cost (₹)": (variable_cost * production).astype(int),
        "Holding cost (₹)": np.array([HOLDING_COST * inventory[t] if t < 11 else 0 for t in range(12)], dtype=int),
        "Setup decision y": setup
    })

    return {
        "result": result,
        "table": solution,
        "production_cost": production_cost,
        "holding_cost": holding_cost,
        "fixed_cost": fixed_cost_total,
        "objective": production_cost + holding_cost + fixed_cost_total,
        "zero_months": months[production == 0].tolist(),
        "production_months": months[production > 0].tolist(),
        "setup_cost_per_month": setup_cost
    }

print("Model function defined.")


Model function defined.


In [3]:
lp_solution = solve_production_plan(setup_cost=None)

display(lp_solution["table"])
print(f"Variable production cost: ₹{lp_solution['production_cost']:,.0f}")
print(f"Inventory holding cost:  ₹{lp_solution['holding_cost']:,.0f}")
print(f"Total objective value:   ₹{lp_solution['objective']:,.0f}")
print("Production quantities by month:", lp_solution["table"]["Production (units)"].tolist())
print("Months with zero production:", lp_solution["zero_months"])


,Month,Demand (units),Production (units),Ending inventory (units),Variable cost (₹),Holding cost (₹),Setup decision y
0,1,1000,10000,9000,2500000,270000,1
1,2,2000,5000,12000,1500000,360000,1
2,3,5000,0,7000,0,210000,0
3,4,4000,0,3000,0,90000,0
4,5,3000,0,0,0,0,0
5,6,500,10000,9500,900000,285000,1
6,7,1000,0,8500,0,255000,0
7,8,8000,0,500,0,15000,0
8,9,4000,10000,6500,2000000,195000,1
9,10,2000,1500,6000,375000,180000,1


Variable production cost: ₹7,725,000
Inventory holding cost:  ₹1,860,000
Total objective value:   ₹9,585,000
Production quantities by month: [10000, 5000, 0, 0, 0, 10000, 0, 0, 10000, 1500, 0, 3000]
Months with zero production: [3, 4, 5, 7, 8, 11]


In [4]:
solution_200k = solve_production_plan(setup_cost=200_000)

display(solution_200k["table"])
print(f"Variable production cost: ₹{solution_200k['production_cost']:,.0f}")
print(f"Inventory holding cost:  ₹{solution_200k['holding_cost']:,.0f}")
print(f"Fixed/setup cost:        ₹{solution_200k['fixed_cost']:,.0f}")
print(f"Total objective value:   ₹{solution_200k['objective']:,.0f}")
print(f"Number of months with no production: {len(solution_200k['zero_months'])}")
print("Months with no production:", solution_200k["zero_months"])
print("Months with production:", solution_200k["production_months"])


,Month,Demand (units),Production (units),Ending inventory (units),Variable cost (₹),Holding cost (₹),Setup decision y
0,1,1000,10000,9000,2500000,270000,1
1,2,2000,5000,12000,1500000,360000,1
2,3,5000,0,7000,0,210000,0
3,4,4000,0,3000,0,90000,0
4,5,3000,0,0,0,0,0
5,6,500,10000,9500,900000,285000,1
6,7,1000,0,8500,0,255000,0
7,8,8000,0,500,0,15000,0
8,9,4000,10000,6500,2000000,195000,1
9,10,2000,1500,6000,375000,180000,1


Variable production cost: ₹7,725,000
Inventory holding cost:  ₹1,860,000
Fixed/setup cost:        ₹1,200,000
Total objective value:   ₹10,785,000
Number of months with no production: 6
Months with no production: [3, 4, 5, 7, 8, 11]
Months with production: [1, 2, 6, 9, 10, 12]


In [5]:
solution_400k = solve_production_plan(setup_cost=400_000)

display(solution_400k["table"])
print(f"Variable production cost: ₹{solution_400k['production_cost']:,.0f}")
print(f"Inventory holding cost:  ₹{solution_400k['holding_cost']:,.0f}")
print(f"Fixed/setup cost:        ₹{solution_400k['fixed_cost']:,.0f}")
print(f"Total objective value:   ₹{solution_400k['objective']:,.0f}")
print(f"Number of months with no production: {len(solution_400k['zero_months'])}")
print("Months with no production:", solution_400k["zero_months"])
print("Months with production:", solution_400k["production_months"])


,Month,Demand (units),Production (units),Ending inventory (units),Variable cost (₹),Holding cost (₹),Setup decision y
0,1,1000,10000,9000,2500000,270000,1
1,2,2000,5000,12000,1500000,360000,1
2,3,5000,0,7000,0,210000,0
3,4,4000,0,3000,0,90000,0
4,5,3000,0,0,0,0,0
5,6,500,10000,9500,900000,285000,1
6,7,1000,0,8500,0,255000,0
7,8,8000,0,500,0,15000,0
8,9,4000,10000,6500,2000000,195000,1
9,10,2000,1500,6000,375000,180000,1


Variable production cost: ₹7,725,000
Inventory holding cost:  ₹1,860,000
Fixed/setup cost:        ₹2,400,000
Total objective value:   ₹11,985,000
Number of months with no production: 6
Months with no production: [3, 4, 5, 7, 8, 11]
Months with production: [1, 2, 6, 9, 10, 12]


In [6]:
comparison = pd.DataFrame([
    {
        "Model": "Fixed cost ignored",
        "Setup cost/month (₹)": 0,
        "Variable production cost (₹)": lp_solution["production_cost"],
        "Holding cost (₹)": lp_solution["holding_cost"],
        "Total setup cost (₹)": lp_solution["fixed_cost"],
        "Total objective (₹)": lp_solution["objective"],
        "Production months": len(lp_solution["production_months"]),
        "Zero-production months": len(lp_solution["zero_months"]),
        "Months with production": str(lp_solution["production_months"]),
        "Months with zero production": str(lp_solution["zero_months"])
    },
    {
        "Model": "Fixed cost ₹200,000",
        "Setup cost/month (₹)": 200_000,
        "Variable production cost (₹)": solution_200k["production_cost"],
        "Holding cost (₹)": solution_200k["holding_cost"],
        "Total setup cost (₹)": solution_200k["fixed_cost"],
        "Total objective (₹)": solution_200k["objective"],
        "Production months": len(solution_200k["production_months"]),
        "Zero-production months": len(solution_200k["zero_months"]),
        "Months with production": str(solution_200k["production_months"]),
        "Months with zero production": str(solution_200k["zero_months"])
    },
    {
        "Model": "Fixed cost ₹400,000",
        "Setup cost/month (₹)": 400_000,
        "Variable production cost (₹)": solution_400k["production_cost"],
        "Holding cost (₹)": solution_400k["holding_cost"],
        "Total setup cost (₹)": solution_400k["fixed_cost"],
        "Total objective (₹)": solution_400k["objective"],
        "Production months": len(solution_400k["production_months"]),
        "Zero-production months": len(solution_400k["zero_months"]),
        "Months with production": str(solution_400k["production_months"]),
        "Months with zero production": str(solution_400k["zero_months"])
    }
])

display(comparison)


,Model,Setup cost/month (₹),Variable production cost (₹),Holding cost (₹),Total setup cost (₹),Total objective (₹),Production months,Zero-production months,Months with production,Months with zero production
0,Fixed cost ignored,0,7725000.0,1860000.0,0.0,9585000.0,6,6,"[1, 2, 6, 9, 10, 12]","[3, 4, 5, 7, 8, 11]"
1,"Fixed cost ₹200,000",200000,7725000.0,1860000.0,1200000.0,10785000.0,6,6,"[1, 2, 6, 9, 10, 12]","[3, 4, 5, 7, 8, 11]"
2,"Fixed cost ₹400,000",400000,7725000.0,1860000.0,2400000.0,11985000.0,6,6,"[1, 2, 6, 9, 10, 12]","[3, 4, 5, 7, 8, 11]"
